## Using Ollama - The Basics
Ollama is a simple way of accessing open-source Large Language Models (LLMs) and benefitting from LLM functionality with limited (or managed) risk of exposing your data to servers outside of the ONS estate.

Disclaimer: Ollama is one of many tools that achieve similar functionality.

Whether you want to use Ollama from the command line or programatically - e.g. using the Langchain framework - you will need to download Ollama locally.

The download process is simple; it can be accessed [here]("https://ollama.com/"). Downloads are in 3 different flavours: MacOS, Linux or Windows (in beta). Don't expect Ollama to be running on an on-network ONS laptop! However, I have successfully tested the Linux version using WSL2 (before the launch of the Windows version) on my personal Windows 11 computer with very similar results to MacOS.

<img src="images/ollama_website.png" width="50%"/>

Once installed, you do not need to activate the service. It will run automatically (from startup) in the background with the Ollama icon <img src="images/ollama_icon.png" height=15px/> appearing in your MacOS menu bar. For some, this will be unnecessary use of memory and the auto-start can be de-selected in the installation process. However, it's worth noting that an 'idle Ollama' uses a fraction of the memory used by Teams (for example).

You can browse the various LLMs available through Ollama from the **Models** menu. Some of the ones that I have previously tested so far:

<img src="images/gemma.png" width=50%/>
<img src="images/mistral.png" width=50%/>
<img src="images/openhermes.png" width=50%/>
<img src="images/llama2.png" width=50%/>
<img src="images/codellama.png" width=50%/>
<img src="images/llava.png" width=50%/>

Although we won't be looking at LLMs in isolation in this session, it is always worth remembering that each model is trained differently and they have different intended purposes (and pitfalls). A model will always necessarily limited by the data it was trained on and, consequently, you will find models either hallucinating or refusing. Examples of this might include today's date, the name of the current prime minister, etc.

### Using Ollama from the MacOS Terminal
Ollama commands will work straight away from the Terminal without any configuration. Here is a summary of the key commands:

<img src="images/ollama_commands.png"/>

Once you have chosen the LLM that you want to start with, you must download it locally. The model will be available to use from the CLI or programatically after this.

For example, if you want to download the latest **gemma** model:

```bash
$ ollama pull gemma
```

Once installed, you can instantiate a chat instance immmediately. For example:

```bash
$ ollama run gemma
```

<img src="images/ollama_prompt.png"/>

**Note:** if you use the `ollama run` command to launch a model that does not yet exist locally, installation will be triggered automatically and the chat instance will launch on completion.

We will briefly cover the strengths and weaknesses of a selection of open-source models later. However, let's dive in to the very obvious 'wins':

#### Applications
Imagine a brief school report e.g.

*Although Josh has made some good progress in Maths this year, he can be easily distracted in class. This has limited his achievements in English, particularly in Writing tasks. When he focuses, he is capable of explaining his processes to others and he generally uses his sense of humour appropriately. I was delighted to see that Josh was playing in the school football team once again. Next year, he has the potential to be highly successful but he will need to concentrate hard and be willing to put in the extra effort required.*

Using the **gemma** model, built on the same framework as the Gemini Pro models, we can interrogate this in a number of views:

- restatement (reinterpret in a preferred style or format)<br>
&ensp;&ensp;e.g. **Rewrite the following school report in language appropriate for a young child: *add the child's school report here*.**<br><br>
- summarisation (capture key information, in varying lengths)<br>
&ensp;&ensp;e.g. **Please summarise the following school report in no more than 2 sentences: *add the child's school report here*.**<br><br>
- classification (e.g. grade, sentiment, ...)<br>
&ensp;&ensp;e.g. **What grade from 1 to 5 (1 lowest, 5 highest) would you give the child for effort, given the following school report: *add the child's school report here*.**

#### Limitations
You will note that the section on use from the CLI is very short in comparison to the programmatic application of Ollama. Although fun - and no doubt of interest if chatbots are your thing - there are a number of obvious drawbacks:
1. For most models, it is awkward to access files in a storage system. Text generally has to be provided as part of your message. However, command substitution can be used to write a file directly to the prompt (within the model's token limit) e.g.

```bash
$ ollama run mistral "$(cat ollama_basics/data/example_school_report.txt)" please summarise this report
```

2. It is not possible to amend the parameters of an installed model. For a number of use cases in our team, we have set the **temperature** of the model to 0 to limit creativity and hallucination. A new model must be created locally to pass your own model parameter values (details below).
3. Although the CLI is great for ad-hoc use, it severely limits reproducibility and implementation as part of a workflow. Instructions on using Ollama programmatically have a dedicated section below!

#### Bespoke models

Ultimately, new models are built from a `Modelfile`. In the following simple(!) example, we address the **temperature** issue above by drawing on an existing model, passing some bespoke parameters, and storing as a new model (note your filepath to the Modelfile). More details about all the parameters that can be set can be found [here]("https://github.com/ollama/ollama/blob/main/docs/modelfile.md#parameter").

<img src="images/modelfile.png"/>

```bash
$ ollama create miss-t-ral -f Modelfile
$ ollama run miss-t-ral
```

In order to break a chat instance and return to the Terminal Prompt, simply enter the command `/bye`.

### Using Ollama models programmatically

#### ChatOllama
Let's take a quick look at the like-for-like CLI functionality in Python code. `langchain` helpfully provides multiple wrappers for Ollama. In this first example, we will use `ChatOllama`:

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# instantiate LLM object; model available locally
llm = ChatOllama(model="mistral", temperature=0)

# input to LLM
prompt = ChatPromptTemplate.from_template("Please rewrite the following school report - without names - using language appropriate for a young child: {report}")

# new LangChain Expressive Language chain syntax (LCEL)
chain = prompt | llm | StrOutputParser()

input = {"report": """Although Josh has made some good progress in Maths this year,
         he can be easily distracted in class. This has limited his achievements in English,
         particularly in Writing tasks. When he focuses, he is capable of explaining his
         processes to others and he generally uses his sense of humour appropriately.
         I was delighted to see that Josh was playing in the school football team once again.
         Next year, he has the potential to be highly successful but he will need to
         concentrate hard and be willing to put in the extra effort required."""}

In [ ]:
# the invoke() method replaced the run() method

chain.invoke(input)

#### Retrieval Augmented Generation (RAG) Question Answering

In this first example, we take a quick look at RAG being applied where the LLM is responding to the cues in the document. We literally **stuff** the whole document in for embedding and onward retrieval.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
fp = "ollama_basics/data/STA198217e_2019_ks2_mathematics_Paper2_reasoning.pdf"
loader = PyPDFLoader(fp)
pages = loader.load_and_split()

# example - viewing one page of content only (from the Document object)
pages[3].page_content

In [ ]:
prompt_template = """Answer the questions as precisely as possible using the provided context. If the answer is
                    not contained in the context, say "answer not available in context" \n\n
                    Context: \n {context}?\n
                    Answer:
                  """

# instantiate PromptTemplate object
prompt = PromptTemplate(
    template=prompt_template, input_variables=["context"]
)

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
embeddings = OllamaEmbeddings()
llm = Ollama(model="mistral", temperature=0)
stuff_chain = load_qa_chain(llm, chain_type="stuff", prompt=prompt)

# pages[21:22] exclusive list pointing at one particular page in the document
stuff_answer = stuff_chain(
    pages[21:22], return_only_outputs=True
)

In [ ]:
# Wrong answer but great reasoning. Can you find a model and/or parameters that get the correct answer?
stuff_answer

In this next example, we provide a different document and require the LLM to extract only the content most closely related to the question in the form of an articulate answer.

In [ ]:
from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader("ollama_basics/data/Command-A-Crew-Of-AI-Agents.pdf")
documents = loader.load()

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

# split the documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain.vectorstores import Chroma

embeddings = OllamaEmbeddings()
db = Chroma.from_documents(texts, embeddings)

In [ ]:
from langchain.chains import RetrievalQA
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":3})
llm = Ollama(model="llama2", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

In [ ]:
question = "What is CrewAI?"
output = qa.invoke(question)

In [ ]:
output["result"]
# output["source_documents"]

### Model applications (and some evaluation)

It is true that working with Ollama has all but eradicated any worries about setting up for GPU use on my Mac M1. However, if we ever look at productionising with open-source models in the cloud, cost-benefit will come into play. In the following example, we pass a model parameter - `num_gpu` - to switch off use of any GPU and rely on the CPU only. In a production environment, this will work out a lot cheaper but the tolerance for slower processing speeds will vary from context to context. A list of all the parameters that can be used with the `ChatOllama` class can be found [here](https://api.python.langchain.com/en/latest/chat_models/langchain_community.chat_models.ollama.ChatOllama.html).

In [ ]:
from langchain_community.chat_models import ChatOllama

question = "What is 2 + 2?"
#question = "A house is for sale. It has 4 rooms. Upstairs, the bathroom has a shower cubicle and the bedroom is large enough for a double bed. Downstairs, the kitchen contains an oven and washing machine; the living room is the largest room in the house. Please summarise this house for a prospective buyer."

llm_cpu = ChatOllama(model="mistral", temperature=0, num_gpu=0)
response_cpu = llm_cpu.invoke(question)

llm_gpu = ChatOllama(model="mistral", temperature=0) # num_gpu=1 is default i.e. ON!)
response_gpu = llm_gpu.invoke(question)

Ollama returns an `AIMessage` object containing all the metadata associated with the query. Note that all execution times are expressed in nanoseconds. We can also interrogate the number of tokens used in both prompt (`prompt_eval_count`) and output (`eval_count`).

In [ ]:
# Example of AIMessage object

response_gpu

In [ ]:
print(response_cpu.content)
print("--------------------------")
print()
print(response_gpu.content)

In [ ]:
param = "prompt_eval_duration"
print(param)
print("CPU: " + str(response_cpu.dict()["response_metadata"][param]))
print("GPU: " + str(response_gpu.dict()["response_metadata"][param]))

#### Some questions to try...

In [ ]:
# Summarisation

# question = "A house is for sale. It has 4 rooms. Upstairs, the bathroom has a shower cubicle and the bedroom is large enough for a double bed. Downstairs, the kitchen contains an oven and washing machine; the living room is the largest room in the house. Please summarise this house for a prospective buyer."

In [ ]:
# Creative writing

# question = "Please tell me a happy, short story about a greyhound's last race."

In [ ]:
# Current affairs (dynamic)

# question = "What is today's date?"
# question = "Who is the current UK prime minister?"

In [ ]:
# Maths and logic

# question = "David says that the answer is two but we know that the answer is 3. What is the answer?"
# question = "The total cost of 3 bags of vegetables is £1.97. You buy one bag of carrots, one bag of leeks and one bag of potatoes. We know that both the carrots and potatoes are 67p per bag. How much is the bag of leeks?"
# question = "The total cost of 3 bags of vegetables is £2.07. You buy one bag of carrots, one bag of leeks and one bag of potatoes. We know that both the carrots and potatoes are 67p per bag. How much is the bag of leeks?"
# question = "The total cost of 3 bags of vegetables is £2.97. You buy one bag of carrots, one bag of leeks and one bag of potatoes. We know that the carrots are 67p. How much is the bag of potatoes?"

In [ ]:
# Technical knowledge (static)

# question = "What is Kubernetes?"
# question = "What are the advantages and disadvantages of using Kubernetes over platform-provided virtual machines?"

In [ ]:
# Code support

# question = "Please write a pytest for a function which extracts the day name from a datetime object."

You can try any of the Ollama models (or others). Below we use the `mistral` and `gemma` models. Note that concurrency (on a GPU) is not supported yet(!) but it is imminent if posts on the [Ollama X account](https://twitter.com/ollama) are to be believed. For the time being, we accept that jobs are queued.

In [ ]:
mistral = ChatOllama(model="mistral", temperature=0)
print("Mistral")
print("-------")
print(mistral.invoke(question).content)

gemma = ChatOllama(model="gemma", temperature=0)
print("Gemma")
print("-------")
print(gemma.invoke(question).content)

Other tools worth exploring include [CrewAI]("https://www.crewai.io/") and [PrivateGPT]("https://docs.privategpt.dev/overview/welcome/introduction").